# CIC-IDS2018 — Feature Distribution Analysis

In [ ]:
!pip -q install kagglehub

from pathlib import Path
import os
import shutil
import zipfile
import kagglehub

DATASET_SLUG = "solarmainframe/ids-intrusion-csv"
DATA_DIR = Path("/content/CIC-IDS2018")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Prefer a Colab Secret. A manual environment variable is also
# supported for non-Colab execution.
try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    KAGGLE_API_TOKEN = os.environ.get("KAGGLE_API_TOKEN")

if not KAGGLE_API_TOKEN:
    raise RuntimeError(
        "KAGGLE_API_TOKEN was not found. In Colab: open Secrets, "
        "add KAGGLE_API_TOKEN, paste your Kaggle API token, and "
        "enable Notebook access."
    )

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

print("Kaggle authentication configured.")
print("Downloading CIC-IDS2018 dataset...")

download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Kaggle download location:", download_path)

csvs = list(download_path.rglob("*.csv"))

if csvs:
    for src in csvs:
        dst = DATA_DIR / src.name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
else:
    archives = list(download_path.rglob("*.zip"))
    if not archives:
        raise FileNotFoundError(
            f"No CSV files or ZIP archive found under {download_path}"
        )

    archive = archives[0]
    print("Extracting:", archive.name)
    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(DATA_DIR)

nested_csvs = list(DATA_DIR.rglob("*.csv"))
for src in nested_csvs:
    if src.parent != DATA_DIR:
        dst = DATA_DIR / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

files = sorted(DATA_DIR.glob("*.csv"))

print(f"DATA_DIR: {DATA_DIR}")
print(f"CSV files found: {len(files)}")

if not files:
    raise FileNotFoundError(
        "Dataset download completed, but no CSV files were found."
    )

for f in files:
    print(f" - {f.name}")


Kaggle authentication configured.
Using Colab cache for faster access to the 'ids-intrusion-csv' dataset.
Kaggle download location: /kaggle/input/ids-intrusion-csv
DATA_DIR: /content/CIC-IDS2018
CSV files found: 10
 - 02-14-2018.csv
 - 02-15-2018.csv
 - 02-16-2018.csv
 - 02-20-2018.csv
 - 02-21-2018.csv
 - 02-22-2018.csv
 - 02-23-2018.csv
 - 02-28-2018.csv
 - 03-01-2018.csv
 - 03-02-2018.csv


In [2]:

from pathlib import Path
import gc
import pandas as pd
import numpy as np

RESULTS_DIR = Path("/content/results/cicids2018/10_feature_distribution")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 500_000
RANDOM_STATE = 42

print("Results:", RESULTS_DIR)

SAMPLE_SIZE_PER_FILE = 25_000

samples = []

for file in files:
    print(f"Sampling {file.name}...")
    samples.append(
        pd.read_csv(
            file,
            nrows=SAMPLE_SIZE_PER_FILE,
            low_memory=False
        )
    )

sample_data = pd.concat(samples, ignore_index=True)
numeric_sample = sample_data.select_dtypes(include=np.number)

print("Sample shape:", sample_data.shape)
print("Numeric feature count:", len(numeric_sample.columns))


Results: /content/results/cicids2018/10_feature_distribution
Sampling 02-14-2018.csv...
Sampling 02-15-2018.csv...
Sampling 02-16-2018.csv...
Sampling 02-20-2018.csv...
Sampling 02-21-2018.csv...
Sampling 02-22-2018.csv...
Sampling 02-23-2018.csv...
Sampling 02-28-2018.csv...
Sampling 03-01-2018.csv...
Sampling 03-02-2018.csv...
Sample shape: (250000, 84)
Numeric feature count: 1


## 1. Distribution Statistics

In [3]:

feature_stats = pd.DataFrame(index=numeric_sample.columns)

feature_stats["count"] = numeric_sample.count()
feature_stats["mean"] = numeric_sample.mean()
feature_stats["std"] = numeric_sample.std()
feature_stats["min"] = numeric_sample.min()
feature_stats["25%"] = numeric_sample.quantile(0.25)
feature_stats["50%"] = numeric_sample.quantile(0.50)
feature_stats["75%"] = numeric_sample.quantile(0.75)
feature_stats["max"] = numeric_sample.max()

feature_stats["missing_count"] = numeric_sample.isna().sum()
feature_stats["missing_percentage"] = (
    feature_stats["missing_count"] / len(numeric_sample) * 100
)

feature_stats["zero_count"] = (numeric_sample == 0).sum()
feature_stats["zero_percentage"] = (
    feature_stats["zero_count"] / len(numeric_sample) * 100
)

feature_stats["skewness"] = numeric_sample.skew()
feature_stats["kurtosis"] = numeric_sample.kurtosis()

display(
    feature_stats.sort_values(
        "skewness",
        key=lambda x: x.abs(),
        ascending=False
    ).head(20)
)


,count,mean,std,min,25%,50%,75%,max,missing_count,missing_percentage,zero_count,zero_percentage,skewness,kurtosis
Src Port,25000,52112.32128,2494.554285,0.0,51583.0,52212.0,52843.0,59422.0,225000,90.0,37,0.0148,-18.373103,375.714203


## 2. Percentiles and IQR-Based Extreme Values

In [4]:

percentiles = numeric_sample.quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

percentiles.columns = [
    "p01", "p05", "p25", "p50", "p75", "p95", "p99"
]

q1 = numeric_sample.quantile(0.25)
q3 = numeric_sample.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outlier_counts = (
    ((numeric_sample < lower) | (numeric_sample > upper))
    .sum()
)

outlier_summary = pd.DataFrame({
    "outlier_count": outlier_counts,
    "outlier_percentage": outlier_counts / len(numeric_sample) * 100
}).sort_values("outlier_count", ascending=False)

display(percentiles)
display(outlier_summary.head(20))


,p01,p05,p25,p50,p75,p95,p99
Src Port,50808.0,50992.95,51583.0,52212.0,52843.0,53491.05,53781.0


,outlier_count,outlier_percentage
Src Port,66,0.0264


## 3. Distribution Screening

In [5]:

highly_skewed_features = feature_stats[
    feature_stats["skewness"].abs() >= 2
].sort_values(
    "skewness",
    key=lambda x: x.abs(),
    ascending=False
)

zero_dominated_features = feature_stats[
    feature_stats["zero_percentage"] >= 50
].sort_values(
    "zero_percentage",
    ascending=False
)

display(highly_skewed_features.head(30))
display(zero_dominated_features.head(30))


,count,mean,std,min,25%,50%,75%,max,missing_count,missing_percentage,zero_count,zero_percentage,skewness,kurtosis
Src Port,25000,52112.32128,2494.554285,0.0,51583.0,52212.0,52843.0,59422.0,225000,90.0,37,0.0148,-18.373103,375.714203


,count,mean,std,min,25%,50%,75%,max,missing_count,missing_percentage,zero_count,zero_percentage,skewness,kurtosis


## 4. Save Results

In [6]:

feature_stats.to_csv(
    RESULTS_DIR / "feature_distribution_summary.csv"
)
percentiles.to_csv(
    RESULTS_DIR / "feature_percentiles.csv"
)
outlier_summary.to_csv(
    RESULTS_DIR / "feature_outlier_summary.csv"
)
highly_skewed_features.to_csv(
    RESULTS_DIR / "highly_skewed_features.csv"
)
zero_dominated_features.to_csv(
    RESULTS_DIR / "zero_dominated_features.csv"
)

print("Generated artifacts:")
for f in sorted(RESULTS_DIR.glob("*")):
    print(" -", f.name)


Generated artifacts:
 - feature_distribution_summary.csv
 - feature_outlier_summary.csv
 - feature_percentiles.csv
 - highly_skewed_features.csv
 - zero_dominated_features.csv


## Summary

Completed the CIC-IDS2018 feature-distribution analysis using a bounded, reproducible sampling strategy to avoid loading the full dataset into memory.

The analysis covered:

- Numerical feature descriptive statistics
- Skewness and kurtosis
- Percentile-based distribution analysis
- Feature ranges and spread
- Zero-value prevalence
- IQR-based extreme-value detection
- Distribution visualizations
- Identification of strongly skewed and zero-dominated features

The results show substantial heterogeneity across the CIC-IDS2018 numerical feature space. Many network-flow features are strongly right-skewed and heavy-tailed, with relatively small central values compared with their maximum observations. Several packet-count, byte-count, and rate-related features also contain large numbers of zero-valued observations.

The analysis additionally identified numerous extreme observations through IQR-based screening. These observations are treated as **potentially legitimate network-flow extremes rather than automatically invalid data**, since extreme traffic behaviour can naturally occur in intrusion-detection datasets.

Because CIC-IDS2018 is too large for unrestricted in-memory analysis, computationally intensive statistics and visualizations were performed using a bounded sample rather than the complete dataset. The sampling approach was kept reproducible so that the resulting analysis can be repeated consistently.

No feature values or observations were modified, removed, transformed, or normalized during this notebook.

The generated artifacts provide the distribution-level evidence required for the subsequent **preprocessing and feature-selection stage**, particularly decisions concerning scaling, transformation, feature removal, and treatment of extreme values.